In [56]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance
import os
import fs
import process_data as dp
import mdp_utils
from scipy import stats
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [57]:
data_folder = 'C:\\Users\\shirl\\Documents\\Studie\\2025-2026\\Thesis\\personalized-coping-challenges\\data'
results_folder = 'C:\\Users\\shirl\\Documents\\Studie\\2025-2026\\Thesis\\personalized-coping-challenges\\results\\'

num_actions = 4
max_count = 0

# Load data
action_data = pd.read_csv(os.path.join(data_folder, 'challenge_info.csv'))
samples = pd.read_csv(os.path.join(data_folder, 'processed_samples.csv'))


cluster_vars = ['likedness', 'usefulness', 'difficulty']
actions_clustered, _, cluster_cols = dp.cluster_actions(action_data, cluster_vars, num_clusters=num_actions)
actions_clustered['cat_diff_cluster'] = actions_clustered.groupby(['category', 'expert_score']).ngroup()

cluster_col = 'cluster_all'

reward_cols = ["r_time","r_likedness", "r_usefulness", "r_expert"]
weights = [1/len(reward_cols)] * len(reward_cols)  # Equal weights for all objectives

The features we want to choose from

In [58]:
possible_state_features = ['TIR', 'TIME_Q', 'GOOD', 'MOT']


In [59]:
# We want to test multiple weight combinations
weights_list = np.random.dirichlet(np.ones(len(reward_cols)), size=20)  

In [60]:
# Tiredness and time available are fixed
fixed_features = ['TIR', 'TIME_Q']

num_vals_per_feature = [2, 2, 2]
selected_with_fixed_multiple_weights = fs.feature_selection_with_fixed_multiple_weights(samples, actions_clustered, possible_state_features, fixed_features,
                                                        reward_cols, cluster_col=cluster_col, weights_list=weights_list, num_act=num_actions, num_vals_per_selected_feature=num_vals_per_feature,
                                                        discount_factor=0.7, scalarization='linear', max_count=max_count)

['TIR', 'TIME_Q', 'GOOD'] [2, 2, 2]
Number of samples after processing: 3111
['TIR', 'TIME_Q', 'MOT'] [2, 2, 2]
Number of samples after processing: 3111
Candidate avg p-values: [np.float64(0.017554159326928977), np.float64(0.0013733838389952929)]
Candidate avg F-stats: [np.float64(13.08492983827827), np.float64(18.315200067639672)]
Added: MOT (Avg p-value: 0.0013733838389952929)


In [61]:
num_vals_per_feature = [3, 3, 3]
selected_with_fixed_multiple_weights = fs.feature_selection_with_fixed_multiple_weights(samples, actions_clustered, possible_state_features, fixed_features,
                                                        reward_cols, cluster_col=cluster_col, weights_list=weights_list, num_act=num_actions, num_vals_per_selected_feature=num_vals_per_feature,
                                                        discount_factor=0.7, scalarization='linear', max_count=max_count)

['TIR', 'TIME_Q', 'GOOD'] [3, 3, 3]
Number of samples after processing: 3111
['TIR', 'TIME_Q', 'MOT'] [3, 3, 3]
Number of samples after processing: 3111
Candidate avg p-values: [np.float64(8.970017199490129e-06), np.float64(1.7154736879333942e-06)]
Candidate avg F-stats: [np.float64(26.35062504166545), np.float64(24.573451731851385)]
Added: MOT (Avg p-value: 1.7154736879333942e-06)


In [62]:
binning_combinations = [[2,2,2], [3,3,3], [2,3,2], [3,2,3], [2,2,3], [3,3,2]]

optimal_binning, results = fs.bin_selection_manual_combinations(
    df=samples,
    actions_clustered=actions_clustered,
    selected_features=['TIR', 'TIME_Q', 'MOT'],
    reward_cols=reward_cols,
    cluster_col='cluster_all',
    num_act=num_actions,
    binning_combinations=binning_combinations,  # or binning_combinations_dict
    max_count=max_count,
    discount_factor=0.7,
    scalarization='linear',
    n_weight_samples=10,
    seed=42,
    min_samples_per_state=10
)

Testing 6 binning combinations
Features (in order): ['TIR', 'TIME_Q', 'MOT']
Using 10 random weight vectors


COMBINATION 1/6
Configuration: {'TIR': 2, 'TIME_Q': 2, 'MOT': 2}
State space size: 8
Number of samples after processing: 3111
Coverage: 100.00% (8/8 states)
Samples per state: 388.9

Evaluating across 10 weight vectors...

Results:
  Mean F-stat: 9.21 ± 5.59
  Mean return: 1.7046 ± 0.5063
  Aggregate score: 9.21

  F-stats by feature (mean across weights):
    TIR: 0.16 ± 0.15
    TIME_Q: 3.36 ± 0.99
    MOT: 24.11 ± 15.91
  ✓ NEW BEST!

COMBINATION 2/6
Configuration: {'TIR': 3, 'TIME_Q': 3, 'MOT': 3}
State space size: 27
Number of samples after processing: 3111
Coverage: 100.00% (27/27 states)
Samples per state: 115.2

Evaluating across 10 weight vectors...

Results:
  Mean F-stat: 10.95 ± 5.60
  Mean return: 1.7843 ± 0.5167
  Aggregate score: 10.95

  F-stats by feature (mean across weights):
    TIR: 0.37 ± 0.20
    TIME_Q: 2.35 ± 0.77
    MOT: 30.15 ± 16.40
  ✓ NEW BEST!

C